# 🏗️ 00 · 大数据体系与 HDFS

> 大数据面试第一题常是"数据湖/数仓/Hadoop 体系"。本节搭**大数据技术全景**框架，
> 再用**模拟实现**讲清 **HDFS 的块/副本/读写流程**——不依赖集群，numpy 就能跑。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 大数据技术栈有哪些层？ | 采集/存储/计算/调度/OLAP |
| 2 | HDFS 怎么存大文件？ | 分块 + 副本 |
| 3 | 读写怎么走？ | NameNode 元数据 + DataNode 数据 |
| 4 | 副本策略为什么重要？ | 容错 3 副本 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 大数据技术栈全景（必背框架）

In [2]:
stack = [
    ('采集', 'Flume / Kafka / Logstash / DataX / Sqoop'),
    ('存储', 'HDFS / HBase / Kudu / Delta / Iceberg / Hudi'),
    ('计算', 'MapReduce / Spark / Flink / Hive / Presto / StarRocks'),
    ('调度', 'Airflow / DolphinScheduler / YARN / K8s'),
    ('查询', 'Presto / Trino / Impala / Doris / ClickHouse'),
    ('治理', '数仓分层 / 元数据 / 数据质量 / 数据安全'),
]
for layer, tools in stack:
    print(f'  {layer:<4} | {tools}')
print('\n➡️ 面试串讲：先说全貌，再说"我负责的模块"，最后讲联动')

  采集   | Flume / Kafka / Logstash / DataX / Sqoop
  存储   | HDFS / HBase / Kudu / Delta / Iceberg / Hudi
  计算   | MapReduce / Spark / Flink / Hive / Presto / StarRocks
  调度   | Airflow / DolphinScheduler / YARN / K8s
  查询   | Presto / Trino / Impala / Doris / ClickHouse
  治理   | 数仓分层 / 元数据 / 数据质量 / 数据安全

➡️ 面试串讲：先说全貌，再说"我负责的模块"，最后讲联动


## 2️⃣ 存储 vs 计算分离思想

In [3]:
print('''存算分离（存储计算分离）：
  HDFS = 存储（DataNode 只存数据）
  Spark/Flink = 计算（用完即释放，不持有数据）
  好处：弹性扩缩容、成本低、统一存储（湖仓一体基础）；
  代价：网络 IO 变大 → 数据本地性优化（把任务调度到数据所在节点）。
''')

存算分离（存储计算分离）：
  HDFS = 存储（DataNode 只存数据）
  Spark/Flink = 计算（用完即释放，不持有数据）
  好处：弹性扩缩容、成本低、统一存储（湖仓一体基础）；
  代价：网络 IO 变大 → 数据本地性优化（把任务调度到数据所在节点）。



## 3️⃣ HDFS 分块：把大文件切成块

In [4]:
# 模拟一个大文件，按 128MB 块大小分块（这里用字符数模拟）
file_size_mb = 500
block_size_mb = 128
n_blocks = (file_size_mb + block_size_mb - 1) // block_size_mb
sizes = [block_size_mb] * (n_blocks - 1) + [file_size_mb - block_size_mb * (n_blocks - 1)]
print(f'文件 {file_size_mb}MB：块大小 {block_size_mb}MB -> 分成 {n_blocks} 个块，末块 {sizes[-1]}MB')
assert sum(sizes) == file_size_mb
# 模拟块数据（字节数组）
block_data = [np.random.default_rng(i).integers(0, 256, 64, dtype=np.uint8) for i in range(n_blocks)]
print(f'每个块 64 字节样例：块0 = {block_data[0][:8]}...')
print('➡️ 真实 HDFS 默认块 128MB/256MB，块是分布式存储与容错的基本单位')

文件 500MB：块大小 128MB -> 分成 4 个块，末块 116MB
每个块 64 字节样例：块0 = [ 95 130 194 217 207 235  15 163]...
➡️ 真实 HDFS 默认块 128MB/256MB，块是分布式存储与容错的基本单位


## 4️⃣ 副本机制：3 副本容错

In [5]:
# 模拟副本：每块存 3 份在不同"节点"
n_nodes = 5
replication = 3
assign = {}
for b in range(n_blocks):
    nodes = list(rng.choice(n_nodes, size=replication, replace=False))
    assign[b] = nodes
    print(f'块{b}: 副本 -> 节点{nodes}')

# 容错演示：挂掉 1-2 个节点数据是否仍完整
for fail in [[0], [0, 1], [0, 1, 2]]:
    ok = True
    for b in range(n_blocks):
        if not set(assign[b]) & set(range(n_nodes)) - set(fail):
            ok = False
    print(f'节点 {fail} 宕机 -> 数据可读: {ok}')
print('\n➡️ 3 副本=块丢失率极低；面试点：副本放置策略（同机架/跨机架）')

块0: 副本 -> 节点[np.int64(4), np.int64(0), np.int64(3)]
块1: 副本 -> 节点[np.int64(0), np.int64(3), np.int64(2)]
块2: 副本 -> 节点[np.int64(1), np.int64(3), np.int64(4)]
块3: 副本 -> 节点[np.int64(3), np.int64(2), np.int64(0)]
节点 [0] 宕机 -> 数据可读: True
节点 [0, 1] 宕机 -> 数据可读: True
节点 [0, 1, 2] 宕机 -> 数据可读: True

➡️ 3 副本=块丢失率极低；面试点：副本放置策略（同机架/跨机架）


## 5️⃣ 读写流程（NameNode/DataNode 分工）

In [6]:
print('''读流程：
  1. Client 问 NameNode：文件在哪些块、各块在哪些 DataNode（元数据）
  2. Client 就近（机架感知）从 DataNode 拉块
  3. 拼装成完整文件

写流程：
  1. Client 问 NameNode 申请创建文件
  2. 按块依次写：写块 -> 流式复制到副本节点（pipeline）
  3. 全部写完 -> 告知 NameNode 提交

关键：NameNode 只管元数据（内存），数据流不过 NameNode（否则瓶颈）。
NameNode 单点 -> HA（Active/Standby + JournalNode）。
''')

读流程：
  1. Client 问 NameNode：文件在哪些块、各块在哪些 DataNode（元数据）
  2. Client 就近（机架感知）从 DataNode 拉块
  3. 拼装成完整文件

写流程：
  1. Client 问 NameNode 申请创建文件
  2. 按块依次写：写块 -> 流式复制到副本节点（pipeline）
  3. 全部写完 -> 告知 NameNode 提交

关键：NameNode 只管元数据（内存），数据流不过 NameNode（否则瓶颈）。
NameNode 单点 -> HA（Active/Standby + JournalNode）。



## 6️⃣ 面试高频：HDFS 相关速答

In [7]:
print('''1) 为什么块大（128MB）？  -> 减少元数据量、减少寻道开销；
2) 小文件问题？           -> 每个文件有元数据，海量小文件压垮 NameNode；
3) 解决小文件？           -> 合并（SequenceFile/Har）、分区归档、Iceberg 等湖格式；
4) HDFS 适合什么？        -> 大文件批量读写；不适合秒级随机访问/低延迟；
5) 与对象存储对比？       -> HDFS 块语义 vs OSS 桶/对象语义。
''')
print('✅ 五个高频点已覆盖')

1) 为什么块大（128MB）？  -> 减少元数据量、减少寻道开销；
2) 小文件问题？           -> 每个文件有元数据，海量小文件压垮 NameNode；
3) 解决小文件？           -> 合并（SequenceFile/Har）、分区归档、Iceberg 等湖格式；
4) HDFS 适合什么？        -> 大文件批量读写；不适合秒级随机访问/低延迟；
5) 与对象存储对比？       -> HDFS 块语义 vs OSS 桶/对象语义。

✅ 五个高频点已覆盖


## 📝 自测清单

- [ ] 能画出大数据技术栈全景图（六层）
- [ ] 能解释存储计算分离的好处
- [ ] 能手算文件分块数与副本分配
- [ ] 能画出 HDFS 读写流程图
- [ ] 能答 5 个 HDFS 高频问题